In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [2]:
import pandas as pd

train = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip",
    header=0,
    delimiter="\t",
    quoting=3
)

test = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip",
    header=0,
    delimiter="\t",
    quoting=3
)

unlabeled_train = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip",
    header=0,
    delimiter="\t",
    quoting=3
)

print("train:", train.shape)
print("test:", test.shape)
print("unlabeled_train:", unlabeled_train.shape)

train: (25000, 3)
test: (25000, 2)
unlabeled_train: (50000, 2)


In [3]:
import pandas as pd
import os
from nltk.corpus import stopwords
import nltk.data
import logging
import numpy as np  # Make sure that numpy is imported
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier

# from KaggleWord2VecUtility import KaggleWord2VecUtility

In [4]:
def makeFeatureVec(words, model, num_features):
    # Function to average all of the word vectors in a given
    # paragraph

    # Pre-initialize an empty numpy array
    featureVec = np.zeros((num_features,), dtype="float32")

    nwords = 0.

    # Modern gensim vocabulary
    index2word_set = set(model.wv.index_to_key)

    # Loop over each word in the review
    for word in words:
        if word in index2word_set:
            nwords = nwords + 1.
            featureVec = np.add(featureVec, model.wv[word])

    # Divide the result by the number of words to get the average
    featureVec = np.divide(featureVec, nwords)

    return featureVec

In [5]:
def getAvgFeatureVecs(reviews, model, num_features):
    # Given a set of reviews (each one a list of words), calculate
    # the average feature vector for each one and return a 2D numpy array

    # Initialize a counter
    counter = 0.

    # Preallocate a 2D numpy array, for speed
    reviewFeatureVecs = np.zeros(
        (len(reviews), num_features),
        dtype="float32"
    )

    # Loop through the reviews
    for review in reviews:

        # Print a status message every 1000th review
        if counter % 1000. == 0.:
            print("Review %d of %d" % (counter, len(reviews)))

        # Call makeFeatureVec()
        reviewFeatureVecs[int(counter)] = makeFeatureVec(
            review,
            model,
            num_features
        )

        # Increment the counter
        counter = counter + 1.

    return reviewFeatureVecs

In [6]:
import re
from bs4 import BeautifulSoup
from nltk.corpus import stopwords

def review_to_wordlist(review, remove_stopwords=False):
    # 1. Remove HTML
    review_text = BeautifulSoup(review, "html.parser").get_text()

    # 2. Remove non-letters
    review_text = re.sub("[^a-zA-Z]", " ", review_text)

    # 3. Convert words to lower case and split them
    words = review_text.lower().split()

    # 4. Optionally remove stop words
    if remove_stopwords:
        stops = set(stopwords.words("english"))
        words = [w for w in words if w not in stops]

    # 5. Return a list of words
    return words

In [7]:
def review_to_sentences(review, tokenizer, remove_stopwords=False):

    # 1. Split the paragraph into sentences
    raw_sentences = tokenizer.tokenize(review.strip())

    # 2. Loop over each sentence
    sentences = []

    for raw_sentence in raw_sentences:

        # Skip empty sentences
        if len(raw_sentence) > 0:

            # Convert each sentence to a list of words
            sentences.append(
                review_to_wordlist(
                    raw_sentence,
                    remove_stopwords
                )
            )

    # Return a list of sentences
    return sentences

In [8]:
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

In [9]:
print("train:", "train" in globals())
print("unlabeled_train:", "unlabeled_train" in globals())
print("tokenizer:", "tokenizer" in globals())
print("review_to_sentences:", "review_to_sentences" in globals())

train: True
unlabeled_train: True
tokenizer: True
review_to_sentences: True


In [10]:
sentences = []

print("Parsing sentences from training set")

for review in train["review"]:
    sentences += review_to_sentences(review, tokenizer)

print("Parsing sentences from unlabeled set")

for review in unlabeled_train["review"]:
    sentences += review_to_sentences(review, tokenizer)

Parsing sentences from training set


/tmp/ipykernel_16/2421574581.py:7: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  review_text = BeautifulSoup(review, "html.parser").get_text()


Parsing sentences from unlabeled set


In [11]:
print("Number of sentences:", len(sentences))
print(sentences[0][:20])

Number of sentences: 796172
['with', 'all', 'this', 'stuff', 'going', 'down', 'at', 'the', 'moment', 'with', 'mj', 'i', 've', 'started', 'listening', 'to', 'his', 'music', 'watching', 'the']


In [12]:
num_features = 300
min_word_count = 40
num_workers = 4
context = 10
downsampling = 1e-3

print("Training model...")

model = Word2Vec(
    sentences,
    workers=num_workers,
    vector_size=num_features,   # 原教程是 size=num_features
    min_count=min_word_count,
    window=context,
    sample=downsampling,
    seed=1
)

print("Training finished.")

Training model...
Training finished.


In [13]:
print(model.wv["movie"].shape)

(300,)


In [14]:
print(model.wv["movie"][:10])

[-0.4220292  -0.88514894  0.6381664  -0.15779415 -1.6300259   0.7129346
 -0.20132548 -0.12341794  0.35840935  0.7672784 ]


In [15]:
model_name = "300features_40minwords_10context"
model.save(model_name)

print(model.wv.doesnt_match("man woman child kitchen".split()))
print(model.wv.doesnt_match("france england germany berlin".split()))
print(model.wv.doesnt_match("paris berlin london austria".split()))

print(model.wv.most_similar("man"))
print(model.wv.most_similar("queen"))
print(model.wv.most_similar("awful"))

kitchen
berlin
paris
[('woman', 0.5886895656585693), ('lady', 0.5855469703674316), ('lad', 0.5405784845352173), ('millionaire', 0.5310074687004089), ('guy', 0.5118799805641174), ('monk', 0.5070171356201172), ('soldier', 0.5053162574768066), ('businessman', 0.5004892945289612), ('person', 0.49923914670944214), ('farmer', 0.4967697858810425)]
[('princess', 0.6664038300514221), ('showgirl', 0.6045365333557129), ('bride', 0.5955963134765625), ('mistress', 0.5936765670776367), ('maid', 0.5924365520477295), ('belle', 0.5758110284805298), ('prince', 0.5752727389335632), ('stepmother', 0.5730750560760498), ('latifah', 0.5694419741630554), ('countess', 0.5646542906761169)]
[('terrible', 0.7475845217704773), ('horrible', 0.7226264476776123), ('dreadful', 0.7121005058288574), ('abysmal', 0.7108035087585449), ('atrocious', 0.7014555931091309), ('horrendous', 0.6688806414604187), ('appalling', 0.656951367855072), ('horrid', 0.6494035720825195), ('amateurish', 0.607090175151825), ('embarrassing', 0.

In [16]:
def getCleanReviews(reviews):
    clean_reviews = []

    for review in reviews["review"]:
        clean_reviews.append(
            review_to_wordlist(
                review,
                remove_stopwords=True
            )
        )

    return clean_reviews

In [17]:
print("Creating average feature vecs for training reviews")

clean_train_reviews = getCleanReviews(train)

trainDataVecs = getAvgFeatureVecs(
    clean_train_reviews,
    model,
    num_features
)

Creating average feature vecs for training reviews
Review 0 of 25000
Review 1000 of 25000
Review 2000 of 25000
Review 3000 of 25000
Review 4000 of 25000
Review 5000 of 25000
Review 6000 of 25000
Review 7000 of 25000
Review 8000 of 25000
Review 9000 of 25000
Review 10000 of 25000
Review 11000 of 25000
Review 12000 of 25000
Review 13000 of 25000
Review 14000 of 25000
Review 15000 of 25000
Review 16000 of 25000
Review 17000 of 25000
Review 18000 of 25000
Review 19000 of 25000
Review 20000 of 25000
Review 21000 of 25000
Review 22000 of 25000
Review 23000 of 25000
Review 24000 of 25000


In [18]:
print("review_to_wordlist:", "review_to_wordlist" in globals())
print("getCleanReviews:", "getCleanReviews" in globals())
print("makeFeatureVec:", "makeFeatureVec" in globals())
print("getAvgFeatureVecs:", "getAvgFeatureVecs" in globals())
print("model:", "model" in globals())
print("num_features:", "num_features" in globals())

review_to_wordlist: True
getCleanReviews: True
makeFeatureVec: True
getAvgFeatureVecs: True
model: True
num_features: True


In [19]:
clean_test_reviews = getCleanReviews(test)

testDataVecs = getAvgFeatureVecs(
    clean_test_reviews,
    model,
    num_features
)

print(testDataVecs.shape)

Review 0 of 25000
Review 1000 of 25000
Review 2000 of 25000
Review 3000 of 25000
Review 4000 of 25000
Review 5000 of 25000
Review 6000 of 25000
Review 7000 of 25000
Review 8000 of 25000
Review 9000 of 25000
Review 10000 of 25000
Review 11000 of 25000
Review 12000 of 25000
Review 13000 of 25000
Review 14000 of 25000
Review 15000 of 25000
Review 16000 of 25000
Review 17000 of 25000
Review 18000 of 25000
Review 19000 of 25000
Review 20000 of 25000
Review 21000 of 25000
Review 22000 of 25000
Review 23000 of 25000
Review 24000 of 25000
(25000, 300)


In [20]:
forest = RandomForestClassifier(
    n_estimators=100
)

print("Fitting a random forest to labeled training data...")

forest = forest.fit(
    trainDataVecs,
    train["sentiment"]
)

Fitting a random forest to labeled training data...


In [21]:
result = forest.predict(testDataVecs)
print(result[:20])

[1 0 1 1 1 1 0 0 0 1 1 1 0 0 0 1 1 1 1 0]


In [22]:
output = pd.DataFrame(
    data={
        "id": test["id"],
        "sentiment": result
    }
)

output.to_csv(
    "/kaggle/working/submission.csv",
    index=False,
    quoting=3
)

In [23]:
import os

print(os.path.exists("/kaggle/working/submission.csv"))
print(output.shape)
print(output.head())

True
(25000, 2)
           id  sentiment
0  "12311_10"          1
1    "8348_2"          0
2    "5828_4"          1
3    "7186_2"          1
4   "12128_7"          1
